# Adversarial Capability Audit

Frozen cooperative actor audit for the 8-ant shared-write checkpoint. The notebook warms the cooperative actor into the two-team adversarial evaluator, keeps both teams frozen, and compares full-write and zero-write controls on the midpoint and sparse layouts.

The default audit is 32 episodes at 500 steps per layout/control/team side. The 2000-step confirm and MP4 renders are opt-in.

In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.35')
if 'jax' in sys.modules:
    print('Restart the kernel before rerunning audit cells so JAX sees the memory settings.')

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PACKAGE_ROOT = PROJECT_ROOT / "src" / "ant_byte_env"
sys.path.insert(0, str(PACKAGE_ROOT.parent))

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / 'runs'
    / 'notebooks'
    / 'exploration_to_forage_proximity_sources_full_layout_50x50_8ants_half_food_2src_shared_writes_from_64env_best'
    / 'checkpoints'
    / 'best_full_layout_proximity_8ants_half_food_shared_writes.pkl'
)
OUTPUT_ROOT = PROJECT_ROOT / 'runs' / 'notebooks' / 'adversarial_capability_audit'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(CHECKPOINT_PATH)
print(OUTPUT_ROOT)

In [ ]:
import argparse
import json
from datetime import datetime, timezone

import jax
import jax.numpy as jnp
import numpy as np
from IPython.display import JSON as DisplayJSON, Video, display
from tqdm.auto import tqdm

from ant_byte_env.experiments import namespace_to_jsonable
from ant_byte_env.training.jax_mappo.adversarial.actions import (
    actions_from_logits,
    validate_action_mode,
)
from ant_byte_env.training.jax_mappo.adversarial.cli import parse_args as parse_adversarial_args
from ant_byte_env.training.jax_mappo.adversarial.env import reset_batch
from ant_byte_env.training.jax_mappo.adversarial.evaluation import evaluate_matrix
from ant_byte_env.training.jax_mappo.adversarial.observations import (
    build_team_actor_observations,
    build_team_central_observations,
)
from ant_byte_env.training.jax_mappo.adversarial.rendering import render_adversarial_rollout
from ant_byte_env.training.jax_mappo.adversarial.rollout import compose_team_actions
from ant_byte_env.training.jax_mappo.adversarial.setup import (
    init_adversarial_params,
    make_env,
)
from ant_byte_env.training.jax_mappo.adversarial.transfer import warm_start_actor_params
from ant_byte_env.training.jax_mappo.checkpointing import read_checkpoint, save_checkpoint
from ant_byte_env.training.jax_mappo.models import get_action_logits
from ant_byte_env.training.jax_mappo.observations import (
    flatten_agent_actions,
    food_observation_scale,
)
from ant_byte_env.training.jax_mappo.updates import init_adam_state


def timestamp_utc():
    return datetime.now(timezone.utc).isoformat()


def json_ready(value):
    if isinstance(value, argparse.Namespace):
        return namespace_to_jsonable(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return path

In [ ]:
RUN_AUDIT = True
RUN_CONFIRM_2000 = True
RUN_RENDERS = True

SEED = 4
WIDTH = 50
HEIGHT = 50
NUM_ANTS_PER_TEAM = 8
ACTOR_VISION_RADIUS = 2
WRITE_BITS = 4
HIDDEN_SIZE = 128
TRAINING_ROLLOUT_TEMPERATURE = 0.75
EVAL_EPISODES = 32
EVAL_MAX_STEPS = 500
CONFIRM_MAX_STEPS = 2000

LAYOUTS = {
    'diagnostic_midpoint': {
        'food_count': 125,
        'food_sources': 4,
        'layout_margin': 6,
        'hub_center_window_size': 0,
        'hub_pair_distance_min': 12,
        'hub_pair_distance_max': 20,
        'food_midpoint_window_size': 8,
    },
    'target_sparse': {
        'food_count': 125,
        'food_sources': 2,
        'layout_margin': 6,
        'hub_center_window_size': 0,
        'hub_pair_distance_min': 20,
        'hub_pair_distance_max': 36,
        'food_midpoint_window_size': 16,
    },
}

BYTE_CONTROLS = {
    'sampled_full_writes': 'sampled_move_sampled_write',
    'greedy_full_writes': 'greedy_move_greedy_write',
    'sampled_zero_writes': 'sampled_move_zero_write',
    'greedy_zero_writes': 'greedy_move_zero_write',
}

for action_mode in BYTE_CONTROLS.values():
    validate_action_mode(action_mode)

audit_config = {
    'created_at': timestamp_utc(),
    'checkpoint': CHECKPOINT_PATH,
    'output_root': OUTPUT_ROOT,
    'eval_episodes': EVAL_EPISODES,
    'eval_max_steps': EVAL_MAX_STEPS,
    'confirm_max_steps': CONFIRM_MAX_STEPS,
    'layouts': LAYOUTS,
    'byte_controls': BYTE_CONTROLS,
    'random_vs_frozen_matrix': True,
}
AUDIT_CONFIG_PATH = write_json(OUTPUT_ROOT / 'audit_config.json', audit_config)
display(DisplayJSON(json_ready(audit_config)))

In [ ]:
def make_eval_args(layout, *, action_mode, max_steps, eval_episodes, seed=SEED):
    argv = [
        '--allow-random-init',
        '--quiet',
        '--seed', str(seed),
        '--total-timesteps', '1',
        '--num-envs', '1',
        '--num-steps', '1',
        '--num-minibatches', '1',
        '--update-epochs', '1',
        '--width', str(WIDTH),
        '--height', str(HEIGHT),
        '--num-ants-per-team', str(NUM_ANTS_PER_TEAM),
        '--max-steps', str(max_steps),
        '--actor-vision-radius', str(ACTOR_VISION_RADIUS),
        '--write-bits', str(WRITE_BITS),
        '--write-while-moving',
        '--random-food',
        '--random-hub',
        '--hidden-size', str(HIDDEN_SIZE),
        '--training-rollout-temperature', str(TRAINING_ROLLOUT_TEMPERATURE),
        '--learner-load-model', str(CHECKPOINT_PATH),
        '--opponent-load-model', str(CHECKPOINT_PATH),
        '--opponent-action-mode', action_mode,
        '--eval-action-mode', action_mode,
        '--eval-episodes', str(eval_episodes),
    ]
    for key, value in layout.items():
        argv.extend([f'--{key.replace("_", "-")}', str(value)])
    return parse_adversarial_args(argv)


def load_frozen_cooperative_policy(args):
    env = make_env(args)
    key = jax.random.PRNGKey(int(args.seed))
    reset_key, init_key = jax.random.split(key)
    _, obs = reset_batch(args=args, env=env, key=reset_key)
    food_scale = food_observation_scale(
        food_count=args.food_count,
        food_sources=getattr(args, 'food_sources', None),
    )
    central_obs = build_team_central_observations(
        obs,
        team=int(args.learner_team),
        num_ants_per_team=int(args.num_ants_per_team),
        food_scale=food_scale,
        write_bits=int(args.write_bits),
    )
    actor_obs = build_team_actor_observations(
        obs,
        team=int(args.learner_team),
        num_ants_per_team=int(args.num_ants_per_team),
        food_scale=food_scale,
        actor_vision_radius=int(args.actor_vision_radius),
        write_bits=int(args.write_bits),
    )
    checkpoint = read_checkpoint(CHECKPOINT_PATH)
    actor_obs_dim = int(actor_obs.shape[-1])
    if int(checkpoint['actor_obs_dim']) != actor_obs_dim:
        raise ValueError(f'checkpoint actor dim {checkpoint["actor_obs_dim"]} != audit actor dim {actor_obs_dim}')
    params = init_adversarial_params(
        init_key,
        args=args,
        central_obs_dim=int(central_obs.shape[-1]),
        actor_obs_dim=actor_obs_dim,
    )
    params = warm_start_actor_params(
        params,
        CHECKPOINT_PATH,
        actor_obs_dim=actor_obs_dim,
        target_write_bits=int(args.write_bits),
    )
    dims = {
        'source_actor_obs_dim': int(checkpoint['actor_obs_dim']),
        'source_central_obs_dim': int(checkpoint['central_obs_dim']),
        'actor_obs_dim': actor_obs_dim,
        'central_obs_dim': int(central_obs.shape[-1]),
        'source_run_name': checkpoint.get('run_name'),
    }
    return env, params, dims


probe_args = make_eval_args(
    LAYOUTS['diagnostic_midpoint'],
    action_mode=BYTE_CONTROLS['greedy_full_writes'],
    max_steps=EVAL_MAX_STEPS,
    eval_episodes=EVAL_EPISODES,
)
_, _, probe_dims = load_frozen_cooperative_policy(probe_args)
probe_dims

In [ ]:
def placement_stats(states):
    hubs = np.asarray(states.hub_pos)[0].astype(np.float32)
    food = np.asarray(states.initial_food)[0]
    food_yx = np.argwhere(food > 0)
    if len(food_yx) == 0:
        food_midpoint_distance = 0.0
    else:
        food_xy = food_yx[:, ::-1].astype(np.float32)
        midpoint = np.mean(hubs, axis=0)
        food_midpoint_distance = float(np.mean(np.sum(np.abs(food_xy - midpoint), axis=1)))
    return {
        'hub_pair_distance': float(np.sum(np.abs(hubs[0] - hubs[1]))),
        'food_midpoint_distance': food_midpoint_distance,
    }


def model_actions(params, obs, *, team, key, args, food_scale):
    actor_obs = build_team_actor_observations(
        obs,
        team=int(team),
        num_ants_per_team=int(args.num_ants_per_team),
        food_scale=food_scale,
        actor_vision_radius=int(args.actor_vision_radius),
        write_bits=int(args.write_bits),
    )
    move_logits, write_logits = get_action_logits(params, actor_obs)
    return actions_from_logits(
        move_logits,
        write_logits,
        key,
        action_mode=str(args.eval_action_mode),
        move_temperature=float(args.training_rollout_temperature),
        write_temperature=float(args.training_rollout_temperature),
    )


def self_play_step(*, env, args, params, learner_team, states, obs, key, food_scale):
    learner_key, opponent_key = jax.random.split(key)
    opponent_team = 1 - int(learner_team)
    learner_actions = model_actions(
        params,
        obs,
        team=int(learner_team),
        key=learner_key,
        args=args,
        food_scale=food_scale,
    )
    opponent_actions = model_actions(
        params,
        obs,
        team=opponent_team,
        key=opponent_key,
        args=args,
        food_scale=food_scale,
    )
    joint_actions = compose_team_actions(
        learner_actions,
        opponent_actions,
        learner_team=int(learner_team),
    )
    nonzero_write_actions = jnp.sum(joint_actions[..., 1] > 0, axis=1)
    states, obs, _, terminated, truncated, infos = jax.vmap(env.step)(
        states,
        flatten_agent_actions(joint_actions),
    )
    return states, obs, terminated, truncated, infos, nonzero_write_actions


def mean_metric(rows, key):
    if not rows:
        return 0.0
    return float(np.mean([row[key] for row in rows]))


def evaluate_control(*, layout_name, control_name, args, env, params, learner_team, seed_offset):
    eval_args = argparse.Namespace(**{**vars(args), 'num_envs': 1})
    food_scale = food_observation_scale(
        food_count=eval_args.food_count,
        food_sources=getattr(eval_args, 'food_sources', None),
    )
    key_base = jax.random.PRNGKey(int(eval_args.seed) + int(seed_offset))
    step_fn = jax.jit(
        lambda current_states, current_obs, action_key: self_play_step(
            env=env,
            args=eval_args,
            params=params,
            learner_team=int(learner_team),
            states=current_states,
            obs=current_obs,
            key=action_key,
            food_scale=food_scale,
        )
    )
    rows = []
    opponent_team = 1 - int(learner_team)
    for episode_index in range(int(eval_args.eval_episodes)):
        episode_key = jax.random.fold_in(key_base, int(episode_index))
        reset_key, rollout_key = jax.random.split(episode_key)
        states, obs = reset_batch(args=eval_args, env=env, key=reset_key)
        placements = placement_stats(states)
        pickup_totals = np.zeros((2,), dtype=np.float32)
        nonzero_write_total = 0.0
        write_slot_total = 0.0
        overwrite_total = 0.0
        episode_length = int(eval_args.max_steps)
        for step_index in range(int(eval_args.max_steps)):
            action_key = jax.random.fold_in(rollout_key, int(step_index))
            states, obs, terminated, truncated, infos, nonzero_write_actions = step_fn(
                states,
                obs,
                action_key,
            )
            pickup_totals += np.asarray(infos.pickup_events)[0].astype(np.float32)
            nonzero_write_total += float(np.asarray(nonzero_write_actions)[0])
            write_slot_total += float(np.asarray(infos.num_writes)[0])
            overwrite_total += float(np.asarray(infos.num_overwrites)[0])
            if bool(np.asarray(terminated)[0]) or bool(np.asarray(truncated)[0]):
                episode_length = step_index + 1
                break
        delivered = np.asarray(states.delivered_food)[0].astype(np.float32)
        final_bytes = np.asarray(states.bytes)[0]
        own = float(delivered[int(learner_team)])
        opponent = float(delivered[opponent_team])
        total_action_slots = max(float(episode_length * 2 * int(eval_args.num_ants_per_team)), 1.0)
        rows.append({
            'layout': layout_name,
            'control': control_name,
            'action_mode': str(eval_args.eval_action_mode),
            'learner_team': int(learner_team),
            'episode': int(episode_index + 1),
            'own_deliveries': own,
            'opponent_deliveries': opponent,
            'total_deliveries': own + opponent,
            'delivery_difference': own - opponent,
            'win': float(own > opponent),
            'own_pickups': float(pickup_totals[int(learner_team)]),
            'opponent_pickups': float(pickup_totals[opponent_team]),
            'total_pickups': float(np.sum(pickup_totals)),
            'episode_length': float(episode_length),
            'nonzero_write_actions': nonzero_write_total,
            'nonzero_write_action_rate': nonzero_write_total / total_action_slots,
            'write_slots': write_slot_total,
            'write_slot_rate': write_slot_total / total_action_slots,
            'overwrites': overwrite_total,
            'final_nonzero_byte_tiles': float(np.sum(final_bytes > 0)),
            'final_byte_sum': float(np.sum(final_bytes.astype(np.float32))),
            'remaining_food': float(np.asarray(states.food)[0].sum()),
            **placements,
        })
    summary_keys = [
        'own_deliveries',
        'opponent_deliveries',
        'total_deliveries',
        'delivery_difference',
        'win',
        'own_pickups',
        'opponent_pickups',
        'total_pickups',
        'episode_length',
        'hub_pair_distance',
        'food_midpoint_distance',
        'nonzero_write_actions',
        'nonzero_write_action_rate',
        'write_slots',
        'write_slot_rate',
        'overwrites',
        'final_nonzero_byte_tiles',
        'final_byte_sum',
        'remaining_food',
    ]
    summary = {f'mean_{key}': mean_metric(rows, key) for key in summary_keys}
    summary.update({
        'episodes': len(rows),
        'max_steps': int(eval_args.max_steps),
        'action_mode': str(eval_args.eval_action_mode),
        'learner_team': int(learner_team),
    })
    return summary, rows

In [ ]:
def run_audit(*, max_steps, eval_episodes, label):
    all_rows = []
    layout_summaries = {}
    total_work_items = len(LAYOUTS) * (len(BYTE_CONTROLS) * 2 + 1)
    progress = tqdm(total=total_work_items, desc=label, leave=True)
    try:
        for layout_index, (layout_name, layout) in enumerate(LAYOUTS.items()):
            layout_args = make_eval_args(
                layout,
                action_mode=BYTE_CONTROLS['greedy_full_writes'],
                max_steps=max_steps,
                eval_episodes=eval_episodes,
            )
            env, params, dims = load_frozen_cooperative_policy(layout_args)
            progress.set_postfix(layout=layout_name, control='matrix_baseline', team='all')
            matrix_metrics = evaluate_matrix(
                params=params,
                opponent_params=params,
                args=layout_args,
                env=env,
            )
            progress.update(1)
            layout_summaries[layout_name] = {
                'layout': dict(layout),
                'dims': dict(dims),
                'args': namespace_to_jsonable(layout_args),
                'matrix_baseline': matrix_metrics,
                'controls': {},
            }
            for control_name, action_mode in BYTE_CONTROLS.items():
                control_args = argparse.Namespace(
                    **{**vars(layout_args), 'eval_action_mode': action_mode, 'opponent_action_mode': action_mode}
                )
                control_payload = {'action_mode': action_mode, 'teams': {}}
                for learner_team in (0, 1):
                    progress.set_postfix(layout=layout_name, control=control_name, team=learner_team)
                    summary, rows = evaluate_control(
                        layout_name=layout_name,
                        control_name=control_name,
                        args=control_args,
                        env=env,
                        params=params,
                        learner_team=learner_team,
                        seed_offset=10_000 * layout_index + 100 * learner_team,
                    )
                    control_payload['teams'][f'learner_team_{learner_team}'] = summary
                    all_rows.extend(rows)
                    progress.update(1)
                layout_summaries[layout_name]['controls'][control_name] = control_payload
    finally:
        progress.close()
    payload = {
        'created_at': timestamp_utc(),
        'checkpoint': CHECKPOINT_PATH,
        'max_steps': int(max_steps),
        'eval_episodes': int(eval_episodes),
        'layouts': LAYOUTS,
        'byte_controls': BYTE_CONTROLS,
        'layout_summaries': layout_summaries,
        'episodes': all_rows,
    }
    output_path = write_json(OUTPUT_ROOT / f'capability_audit_{label}.json', payload)
    return payload, output_path


if RUN_AUDIT:
    audit_500, audit_500_path = run_audit(
        max_steps=EVAL_MAX_STEPS,
        eval_episodes=EVAL_EPISODES,
        label='500step',
    )
else:
    audit_500_path = OUTPUT_ROOT / 'capability_audit_500step.json'
    audit_500 = json.loads(audit_500_path.read_text(encoding='utf-8')) if audit_500_path.exists() else {'status': 'not_run'}

display(DisplayJSON({'artifact': str(audit_500_path), 'layout_summaries': audit_500.get('layout_summaries', {})}))

In [ ]:
if RUN_CONFIRM_2000:
    confirm_2000, confirm_2000_path = run_audit(
        max_steps=CONFIRM_MAX_STEPS,
        eval_episodes=EVAL_EPISODES,
        label='confirm_2000step',
    )
else:
    confirm_2000_path = OUTPUT_ROOT / 'capability_audit_confirm_2000step.json'
    confirm_2000 = json.loads(confirm_2000_path.read_text(encoding='utf-8')) if confirm_2000_path.exists() else {'status': 'not_run'}

display(DisplayJSON({'artifact': str(confirm_2000_path), 'status': confirm_2000.get('status', 'available')}))

In [ ]:
RENDER_LAYOUT = 'diagnostic_midpoint'
RENDER_CONTROLS = ['greedy_full_writes', 'greedy_zero_writes']
RENDER_MAX_FRAMES = 500
RENDER_TILE_SIZE = 16


def write_render_ready_checkpoint(layout_name, *, max_steps):
    layout = LAYOUTS[layout_name]
    args = make_eval_args(
        layout,
        action_mode=BYTE_CONTROLS['greedy_full_writes'],
        max_steps=max_steps,
        eval_episodes=1,
    )
    _, params, dims = load_frozen_cooperative_policy(args)
    checkpoint_path = OUTPUT_ROOT / 'render_checkpoints' / f'{layout_name}_frozen_cooperative.pkl'
    save_checkpoint(
        checkpoint_path,
        params=params,
        opt_state=init_adam_state(params),
        args=args,
        central_obs_dim=int(dims['central_obs_dim']),
        actor_obs_dim=int(dims['actor_obs_dim']),
        run_name='adversarial_capability_audit_frozen_cooperative_render',
        metrics={'source_checkpoint': str(CHECKPOINT_PATH)},
    )
    return checkpoint_path, args


if RUN_RENDERS:
    render_checkpoint_path, render_args = write_render_ready_checkpoint(
        RENDER_LAYOUT,
        max_steps=RENDER_MAX_FRAMES,
    )
    videos = {}
    for control_name in RENDER_CONTROLS:
        action_mode = BYTE_CONTROLS[control_name]
        control_args = argparse.Namespace(
            **{**vars(render_args), 'eval_action_mode': action_mode, 'opponent_action_mode': action_mode}
        )
        video_path = render_adversarial_rollout(
            render_checkpoint_path,
            OUTPUT_ROOT / 'media' / f'{RENDER_LAYOUT}_{control_name}.mp4',
            args=control_args,
            max_frames=RENDER_MAX_FRAMES,
            tile_size=RENDER_TILE_SIZE,
            action_mode=action_mode,
        )
        videos[control_name] = str(video_path)
        display(Video(str(video_path), embed=True))
else:
    videos = {}
    print('Set RUN_RENDERS=True to write MP4 rollouts through render_adversarial_rollout().')

videos